In [2]:
import os
import numpy as np
import pandas as pd


def generate_realized_cov_proxy(
    H,
    in_path="../data/log_returns.csv",
    out_dir="../proxy",
    prefix="realized_cov",
    debug=True
):

    # -----------------------
    # Load returns
    # -----------------------
    df = pd.read_csv(in_path, parse_dates=["Date"]).set_index("Date").sort_index()
    assets = df.columns.tolist()

    R = df.to_numpy()
    dates = df.index
    T, N = R.shape

    # -----------------------
    # Full matrix column names
    # -----------------------
    colnames = [
        f"cov_{assets[i]}__{assets[j]}"
        for i in range(N)
        for j in range(N)
    ]

    rows = []
    out_dates = []

    # -----------------------
    # Compute proxy
    # window = t ... t+H-1
    # -----------------------
    for k in range(T - H + 1):

        window = R[k:k+H]
        Sigma = (window.T @ window) / H

        # enforce symmetry
        Sigma = 0.5 * (Sigma + Sigma.T)

        rows.append(Sigma.flatten())
        out_dates.append(dates[k])

    proxy_df = pd.DataFrame(rows, index=out_dates, columns=colnames)
    proxy_df.index.name = "Date"

    # -----------------------
    # Save
    # -----------------------
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{prefix}_h{H}.csv")
    proxy_df.to_csv(out_path)

    print("Saved:", out_path)
    print("Proxy shape:", proxy_df.shape)
    print("Proxy date range:", proxy_df.index.min().date(), "->", proxy_df.index.max().date())

    # -----------------------
    # Optional sanity check
    # -----------------------
    if debug:
        k = 0
        print("\nSanity check:")
        print("t:", dates[k].date())
        print("window start:", dates[k].date())
        print("window end:", dates[k+H-1].date())

        manual = (R[k:k+H].T @ R[k:k+H]) / H
        stored = proxy_df.iloc[k].to_numpy().reshape(N, N)

        diff = np.abs(manual - stored).max()
        print("max difference manual vs stored:", diff)

    return proxy_df

In [3]:
proxy_21 = generate_realized_cov_proxy(H=21)
proxy_21.head()

Saved: ../proxy/realized_cov_h21.csv
Proxy shape: (2184, 64)
Proxy date range: 2017-03-28 -> 2025-12-02

Sanity check:
t: 2017-03-28
window start: 2017-03-28
window end: 2017-04-26
max difference manual vs stored: 0.0


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__US_EQUITY,cov_INTL_EQUITY__INTL_EQUITY,...,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__US_EQUITY,cov_BTC__INTL_EQUITY,cov_BTC__US_BONDS,cov_BTC__INTL_BONDS,cov_BTC__US_REIT,cov_BTC__INTL_REIT,cov_BTC__GOLD,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2017-03-28,0.000021,0.000016,-0.000005,-0.000002,0.000006,0.000008,-0.000015,0.000032,0.000016,0.000033,...,0.000034,0.000007,0.000032,0.000003,0.000013,0.000004,0.000040,0.000044,0.000007,0.000559
2017-03-29,0.000019,0.000016,-0.000004,-0.000002,0.000004,0.000008,-0.000014,0.000033,0.000016,0.000033,...,0.000034,0.000002,0.000033,0.000003,0.000014,0.000009,0.000041,0.000046,0.000002,0.000597
2017-03-30,0.000019,0.000015,-0.000005,-0.000002,0.000005,0.000009,-0.000015,0.000033,0.000015,0.000033,...,0.000034,0.000003,0.000033,0.000005,0.000015,0.000010,0.000043,0.000047,0.000003,0.000595
2017-03-31,0.000019,0.000016,-0.000005,-0.000002,0.000005,0.000010,-0.000015,0.000045,0.000016,0.000034,...,0.000035,-0.000038,0.000045,0.000018,0.000004,0.000005,0.000067,0.000052,-0.000038,0.000868
2017-04-03,0.000019,0.000016,-0.000005,-0.000002,0.000006,0.000010,-0.000014,0.000049,0.000016,0.000034,...,0.000035,-0.000044,0.000049,0.000027,0.000004,0.000008,0.000054,0.000056,-0.000044,0.000801


In [4]:
proxy_5 = generate_realized_cov_proxy(H=5)
proxy_5.head()

Saved: ../proxy/realized_cov_h5.csv
Proxy shape: (2200, 64)
Proxy date range: 2017-03-28 -> 2025-12-24

Sanity check:
t: 2017-03-28
window start: 2017-03-28
window end: 2017-04-03
max difference manual vs stored: 0.0


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__US_EQUITY,cov_INTL_EQUITY__INTL_EQUITY,...,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__US_EQUITY,cov_BTC__INTL_EQUITY,cov_BTC__US_BONDS,cov_BTC__INTL_BONDS,cov_BTC__US_REIT,cov_BTC__INTL_REIT,cov_BTC__GOLD,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2017-03-28,0.000014,4.679550e-06,-0.000005,-1.073006e-06,7.095865e-06,-0.000005,-0.000011,-0.000045,4.679550e-06,0.000015,...,0.000024,0.000111,-0.000045,-0.000109,0.000056,0.000030,0.000040,0.000053,0.000111,0.001264
2017-03-29,0.000003,4.679550e-06,-0.000002,-8.009377e-08,5.688086e-07,-0.000004,-0.000006,-0.000048,4.679550e-06,0.000015,...,0.000023,0.000109,-0.000048,-0.000109,0.000058,0.000031,0.000040,0.000046,0.000109,0.001281
2017-03-30,0.000004,1.187316e-06,-0.000003,-2.019369e-06,-8.746901e-07,-0.000004,-0.000008,-0.000042,1.187316e-06,0.000011,...,0.000022,0.000111,-0.000042,-0.000106,0.000059,0.000033,0.000046,0.000047,0.000111,0.001283
2017-03-31,0.000004,1.187316e-06,-0.000002,-2.130557e-06,1.227597e-06,0.000001,-0.000005,-0.000009,1.187316e-06,0.000011,...,0.000011,0.000048,-0.000009,-0.000106,0.000054,0.000024,0.000123,0.000087,0.000048,0.001752
2017-04-03,0.000004,2.787166e-07,-0.000002,-2.172700e-06,2.846332e-06,0.000002,-0.000005,0.000009,2.787166e-07,0.000010,...,0.000010,0.000016,0.000009,-0.000085,0.000050,0.000028,0.000081,0.000060,0.000016,0.001383


In [5]:
proxy_10 = generate_realized_cov_proxy(H=10)
proxy_10.head()

Saved: ../proxy/realized_cov_h10.csv
Proxy shape: (2195, 64)
Proxy date range: 2017-03-28 -> 2025-12-17

Sanity check:
t: 2017-03-28
window start: 2017-03-28
window end: 2017-04-10
max difference manual vs stored: 0.0


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__US_EQUITY,cov_INTL_EQUITY__INTL_EQUITY,...,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__US_EQUITY,cov_BTC__INTL_EQUITY,cov_BTC__US_BONDS,cov_BTC__INTL_BONDS,cov_BTC__US_REIT,cov_BTC__INTL_REIT,cov_BTC__GOLD,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2017-03-28,0.000009,1.277174e-06,-0.000003,-1.043685e-06,0.000006,-0.000001,-0.000007,-0.000007,1.277174e-06,0.000009,...,0.000015,0.000031,-0.000007,-0.000059,0.000030,0.000012,0.000060,0.000046,0.000031,0.000908
2017-03-29,0.000003,1.277174e-06,-0.000001,-6.448515e-07,0.000002,-0.000002,-0.000006,-0.000010,1.277174e-06,0.000009,...,0.000031,0.000052,-0.000010,-0.000059,0.000034,0.000013,0.000068,0.000059,0.000052,0.000930
2017-03-30,0.000005,-9.991327e-07,-0.000003,-1.627555e-06,0.000002,-0.000005,-0.000010,-0.000007,-9.991327e-07,0.000007,...,0.000037,0.000050,-0.000007,-0.000056,0.000035,0.000015,0.000072,0.000058,0.000050,0.000927
2017-03-31,0.000009,1.741566e-06,-0.000003,-1.353839e-06,0.000003,-0.000003,-0.000011,0.000014,1.741566e-06,0.000009,...,0.000033,0.000026,0.000014,-0.000045,0.000029,0.000015,0.000081,0.000052,0.000026,0.000979
2017-04-03,0.000016,5.358062e-06,-0.000003,-1.262962e-06,0.000014,0.000007,-0.000014,0.000040,5.358062e-06,0.000010,...,0.000033,0.000004,0.000040,-0.000025,0.000025,0.000017,0.000087,0.000060,0.000004,0.000835


In [6]:
proxy_63 = generate_realized_cov_proxy(H=63)
proxy_63.head()

Saved: ../proxy/realized_cov_h63.csv
Proxy shape: (2142, 64)
Proxy date range: 2017-03-28 -> 2025-10-02

Sanity check:
t: 2017-03-28
window start: 2017-03-28
window end: 2017-06-26
max difference manual vs stored: 0.0


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__US_EQUITY,cov_INTL_EQUITY__INTL_EQUITY,...,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__US_EQUITY,cov_BTC__INTL_EQUITY,cov_BTC__US_BONDS,cov_BTC__INTL_BONDS,cov_BTC__US_REIT,cov_BTC__INTL_REIT,cov_BTC__GOLD,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2017-03-28,0.000019,0.000015,-0.000004,-2.253780e-06,0.000006,0.000010,-0.000011,0.000018,0.000015,0.000030,...,0.000039,0.000032,0.000018,0.000022,-0.000005,-9.879912e-07,0.000008,0.000032,0.000032,0.001742
2017-03-29,0.000019,0.000015,-0.000003,-1.210679e-06,0.000006,0.000011,-0.000011,0.000015,0.000015,0.000030,...,0.000039,0.000035,0.000015,0.000023,-0.000007,-4.729114e-06,0.000005,0.000030,0.000035,0.001755
2017-03-30,0.000020,0.000016,-0.000003,-1.058104e-06,0.000006,0.000011,-0.000011,0.000016,0.000016,0.000030,...,0.000039,0.000035,0.000016,0.000025,-0.000006,-4.107615e-06,0.000006,0.000030,0.000035,0.001756
2017-03-31,0.000021,0.000017,-0.000003,-1.770678e-07,0.000008,0.000012,-0.000010,0.000018,0.000017,0.000031,...,0.000038,0.000035,0.000018,0.000027,-0.000006,-2.825056e-06,0.000009,0.000030,0.000035,0.001756
2017-04-03,0.000021,0.000017,-0.000003,-1.745875e-07,0.000008,0.000012,-0.000010,0.000019,0.000017,0.000031,...,0.000038,0.000033,0.000019,0.000028,-0.000006,-2.774013e-06,0.000006,0.000028,0.000033,0.001735
